# 10 - Expert Routing Telemetry

In Phase 1 (notebooks 01 through 09), I performed per-layer ablations on each layer to construct a basic per-layer quantization policy.  In Phase 2 (notebooks 10 through 14), I will conduct per-expert ablations within the protected layers to see if it is possible to quantize the model even further with minimal performance lost.

As a reminder, in mixture-of-expert systems, only a portion of the experts are activated for each query - in this case, 6 of the 64 expert neurons are activated for a given query.  In this notebook, I will measure which experts are actually used.

**Key question:**
- Are routing patterns concentrated, sparse, task-specific, or broadly distributed?

In [6]:
# add imports here
from pathlib import Path
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer
from IPython.display import Markdown, display

In [8]:
PROJECT_ROOT = Path.cwd().resolve()
RESULTS_DIR = PROJECT_ROOT / "results"
EXPORTS_DIR = PROJECT_ROOT / "exports"

MODEL_ID = "deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct"
REMOTE_CODE_REVISION = "refs/pr/10"
DATASET_ID = "openai/openai_humaneval"

In [10]:
def markdown_table(rows, columns):
    """Render a small list-of-dicts table without adding a pandas dependency."""
    if not rows:
        return "_No rows to display._"

    header = "| " + " | ".join(label for _, label in columns) + " |"
    divider = "| " + " | ".join("---" for _ in columns) + " |"
    body = []

    for row in rows:
        values = []
        for key, _ in columns:
            value = row.get(key, "")
            if isinstance(value, float):
                value = f"{value:.2f}"
            values.append(str(value))
        body.append("| " + " | ".join(values) + " |")

    return "\n".join([header, divider, *body])

In [11]:
config = AutoConfig.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    code_revision=REMOTE_CODE_REVISION,
)

config_rows = []
for attr in [
    "num_hidden_layers",
    "first_k_dense_replace",
    "moe_layer_freq",
    "n_routed_experts",
    "num_experts_per_tok",
    "hidden_size",
    "intermediate_size",
    "moe_intermediate_size",
]:
    config_rows.append({"field": attr, "value": getattr(config, attr, None)})

display(Markdown(markdown_table(config_rows, [("field", "Config Field"), ("value", "Value")])))

| Config Field | Value |
| --- | --- |
| num_hidden_layers | 27 |
| first_k_dense_replace | 1 |
| moe_layer_freq | 1 |
| n_routed_experts | 64 |
| num_experts_per_tok | 6 |
| hidden_size | 2048 |
| intermediate_size | 10944 |
| moe_intermediate_size | 1408 |